# Day 11 · 数据打包与对话模板

**配套讲义**: `days/day-11.md` ｜ **本地可跑**
今天出错的样本，Day 14 会以「loss 不收敛」的形式报复你。

## 1. 先看 Qwen 的 chat template 长什么样

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")
msgs = [{"role": "system", "content": "你是客服"},
        {"role": "user", "content": "<|vision_start|><|image_pad|><|vision_end|>多大码？"},
        {"role": "assistant", "content": "M/L 有现货。"}]
print(tok.apply_chat_template(msgs, tokenize=False))

## 2. 构造带 label mask 的训练样本

In [ ]:
import sys; sys.path.insert(0, "..")
from src.data.build_sft import build_labeled_sample, inspect_sample
sample = {"system": "你是服装店客服", "image": "imgA",
          "user": "多大码有货？", "assistant": "您好，M/L 有现货。"}
enc = build_labeled_sample(sample)
inspect_sample(enc, tok)   # 逐 token 打印：哪些算 loss，哪些 -100

## 3. 三种 mask 错误的反例（今天的考点）

逐个把下面的开关打开跑一遍，看 mask 错成什么样、**后果**是什么。

In [ ]:
WRONG_MODE = None   # 依次改成: "system_in_loss" / "miss_im_end" / "user_in_loss"
# WRONG_MODE = "system_in_loss"   # 模型学会抢答 system
# WRONG_MODE = "miss_im_end"      # 模型学不会停 → 推理时说个没完
# WRONG_MODE = "user_in_loss"     # 模型学会复述问题

if WRONG_MODE:
    enc_bad = build_labeled_sample(sample, wrong_mode=WRONG_MODE)
    inspect_sample(enc_bad, tok)
else:
    print("改 WRONG_MODE 依次观察三种错误。")

## 4. 泄漏检查（图像级）

In [ ]:
from src.data.build_sft import check_leakage
train = [{"image": "imgA"}, {"image": "imgB"}]
eval_ = [{"image": "imgB"}]            # 故意泄漏
n = check_leakage(train, eval_, key=lambda s: s["image"])
print("泄漏数:", n, "（期望 1，被拦住才算检查有效）")

## 5. 全量打包

In [ ]:
# 终端跑：
#   python -m src.data.build_sft --in data/clean/clean.jsonl --out data/processed/
#   python -m src.data.build_sft --inspect data/processed/sft_train.jsonl 0
print("打包命令见注释。跑完把 --inspect 的输出贴到打卡里。")

## 6. 验收
- [ ] 抽 5 条肉眼检查：三种 mask 错误一个没有
- [ ] check_leakage = 0
- [ ] train/eval 意图分布偏差 < 3%